In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmdltf/flyrank-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

A hand-written, readable rule that every later model has to beat. Everything below is **observed** on the starter table (trailing-90-day metrics). Results are **directional** and meant for **decision-support**, not proof.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words**

A content item goes into the refresh-review queue when three things are true together: it was last updated 91 or more days ago, it still shows up in search (300 or more impressions in the trailing 90 days), and its average position is above 3 and no worse than 20, so it sits just below the top results. Flagged items are ranked by impressions, so the pages that more people see come first. An item with no position data (`avg_position = 0`) is never flagged, because 0 means "not measured", not "rank zero".

Score = `stale × visible × in_position_band × impressions_90d`. There are no fitted weights, no label and no trend columns in it. Ranking by impressions is a "size of the prize" proxy, not a prediction.

**Where the cut-offs come from.** They are the data dictionary's own tiers, not numbers fitted to the label: 91+ days is the `91-180` plus `181+` freshness tiers, 300+ impressions is the `moderate` tier and above, and over 3 up to 20 is the `page_1` plus `striking` position tiers. A first draft, written before I had the data, used rounder numbers: 180 days for "stale" (only 174 of 30,000 items are that old, observed, which would leave almost no queue), 500 impressions, and positions 4 to 20. I moved all of these to the dictionary's tiers using the dictionary and the input columns only, before any score was compared with the label, and I changed nothing after seeing the precision numbers. The baseline is frozen from here.

**Two data traps the rule has to survive.** (1) `avg_position = 0` means "no data", and in this table all 1,205 such rows sit in the `top_3` position tier, so that tier column cannot be used to find them. The rule reads `avg_position` directly. (2) `days_since_last_update` is lumpy: 8,773 items sit at exactly 104 days and 11,573 at exactly 20 days (observed), so "stale" mostly means "belongs to the 104-day group".

**Reason codes it can output**

| code | meaning |
|---|---|
| `stale_visible_page1` | flagged; average position over 3 up to 10 |
| `stale_visible_page2` | flagged; average position over 10 up to 20 |
| detail `very_stale` | flagged and last updated 181+ days ago |
| detail `high_visibility` | flagged and 3,000+ impressions |
| `missing_input` | not flagged; impressions, age or position is empty |
| `no_position_data` | not flagged; position is 0 |
| `low_visibility` | not flagged; under 300 impressions |
| `fresh` | not flagged; updated within the last 90 days |
| `position_outside_range` | not flagged; position in the top 3 or worse than 20 |

When several "not flagged" reasons apply, the first one in the order `missing_input → no_position_data → low_visibility → fresh → position_outside_range` is shown.

**Actions it can output:** `review_for_refresh` or `no_action`.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import difflib
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CSV_NAME = Path("/content/drive/MyDrive/data/raw/content_refresh_anonymized.csv")


def find_repo_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / CSV_NAME).exists():
            return p
    raise FileNotFoundError(
        f"Could not find {CSV_NAME} from {here} or any parent folder. "
        "Open the notebook from inside the repo (work/notebooks/) or fix CSV_NAME."
    )


ROOT = find_repo_root()
df = pd.read_csv(ROOT / CSV_NAME)
print("rows, columns:", df.shape)

# The ONLY place column names live. If the data dictionary says a name differs, change it here.
COL = {
    "impressions": "impressions_90d",
    "age_days": "days_since_last_update",
    "position": "avg_position",
    "ctr": "ctr",                    # optional - review notes only (x100 percentage: 0.76 = 0.76%)
    "content_type": "content_type",  # optional - review notes only
    "label_src": "trend_direction",  # evaluation ONLY - the label is (trend_direction == "down"); never enters the score
    "content_id": "content_id",      # pseudonym - ordering/joining only
    "client_id": "client_id",        # pseudonym - grouping only
}
REQUIRED = ["impressions", "age_days", "position", "label_src", "content_id", "client_id"]
missing = [k for k in REQUIRED if COL[k] not in df.columns]
if missing:
    for k in missing:
        print(f"missing '{COL[k]}'  -> closest names in the file: {difflib.get_close_matches(COL[k], list(df.columns), n=3)}")
    raise KeyError(f"Fix the COL map above. Missing: {[COL[k] for k in missing]}")
assert df[COL["content_id"]].is_unique, "content_id is not unique - the grain is not one row per item"

# Cut-offs = the data dictionary's own tiers. Fixed BEFORE any score was compared with the label. Do not tune them against it.
STALE_DAYS = 91          # freshness_tier "91-180" and "181+"  (a 180-day cut-off would flag almost nothing: see cell output)
VERY_STALE_DAYS = 181    # freshness_tier "181+"  (detail tag only)
MIN_IMPRESSIONS = 300    # impression_tier "moderate" and above
HIGH_IMPRESSIONS = 3000  # impression_tier "good" and above  (detail tag only)
POS_TOP3 = 3             # position_tier "top_3" is excluded: already at the top
POS_HI = 20              # position_tier "page_1" + "striking": over 3 up to 20
PAGE1_HI = 10            # over 3 up to 10 -> page1 code, over 10 up to 20 -> page2 code

# Quick profile of the three score inputs
inputs = df[[COL["impressions"], COL["age_days"], COL["position"]]].apply(pd.to_numeric, errors="coerce")
print("empty values per input:", inputs.isna().sum().to_dict())
print("rows with avg_position == 0 (no data):", int((inputs[COL["position"]] == 0).sum()))
display(inputs.describe().round(1))

age = inputs[COL["age_days"]]
print(f"items updated {STALE_DAYS}+ days ago: {int((age >= STALE_DAYS).sum())} | {VERY_STALE_DAYS}+ days ago: {int((age >= VERY_STALE_DAYS).sum())}")
age_counts = age.value_counts()
BULK_AGES = set(age_counts[age_counts >= 0.01 * len(df)].index)   # update ages shared by 1%+ of ALL items
print("update ages shared by 1%+ of all items (rows):", {int(k): int(v) for k, v in age_counts[age_counts >= 0.01 * len(df)].items()})


rows, columns: (30000, 44)
empty values per input: {'impressions_90d': 0, 'days_since_last_update': 0, 'avg_position': 0}
rows with avg_position == 0 (no data): 1205


,impressions_90d,days_since_last_update,avg_position
count,30000.0,30000.0,30000.0
mean,5200.4,46.1,16.3
std,16838.0,42.1,15.2
min,1.0,1.0,0.0
25%,81.0,20.0,6.2
50%,731.0,20.0,10.8
75%,3615.2,104.0,22.3
max,517715.0,373.0,245.0


items updated 91+ days ago: 9345 | 181+ days ago: 174
update ages shared by 1%+ of all items (rows): {20: 11573, 104: 8773, 22: 3564, 8: 1929, 13: 515, 14: 505, 28: 466, 25: 441, 7: 420}


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**How the queue is built**

- Every row gets a score, an action and a reason code. Then every row is ranked, highest score first. Ties keep their original row order, so rows after the last flagged one are **not** meaningfully ordered. Use the `flagged` column to see where the real ranking stops.
- The CSV holds no label and no trend columns, so nothing in it can leak into a later model. To evaluate, join on `content_id`.
- The label is the pipeline's `is_declining_label`, rebuilt here as `trend_direction == "down"` (it is not in the raw CSV). It is built after the ranking and used only for precision@K, always printed next to the base rate. It is only a **proxy** for "needs a refresh".
- With K = 20, one pick moves precision by 0.05. Read small-K numbers as **directional**.

**Result (observed, one run on the 30,000-row table, rule frozen):** 4,483 rows are flagged (15%). The label rate is 0.542 overall, 0.623 inside the flagged set and 0.528 (0.554 once the 1,205 no-position rows are set aside, because they are almost never "down") outside it. At the top of the queue it is weaker: precision@20 = 0.400 (lift 0.74), precision@50 = 0.380, precision@100 = 0.340, all **below** the base rate. Precision stays below the base rate at every K I printed up to 500 (range 0.340 to 0.460) and passes it only when the whole flagged set is counted (0.623, lift 1.15). My reading: the flag itself carries a modest signal, and the impressions ordering does not help at the top.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def rule(d):
    """Transparent baseline. Reads ONLY: impressions, age in days, average position."""
    imp = pd.to_numeric(d[COL["impressions"]], errors="coerce")
    age = pd.to_numeric(d[COL["age_days"]], errors="coerce")
    pos = pd.to_numeric(d[COL["position"]], errors="coerce")

    has_pos = pos.notna() & (pos > 0)              # 0 means "no data", not rank zero
    stale = age >= STALE_DAYS                      # empty age -> False
    visible = imp >= MIN_IMPRESSIONS
    in_band = has_pos & (pos > POS_TOP3) & (pos <= POS_HI)
    missing_in = imp.isna() | age.isna() | pos.isna()

    flagged = stale & visible & in_band
    score = flagged.astype(float) * imp.fillna(0)  # readable on purpose: stale x visible x band x impressions

    reason = np.select(
        [flagged & (pos <= PAGE1_HI), flagged, missing_in, ~has_pos, ~visible, ~stale, ~in_band],
        ["stale_visible_page1", "stale_visible_page2", "missing_input", "no_position_data",
         "low_visibility", "fresh", "position_outside_range"],
        default="not_flagged",
    )
    very_stale = (flagged & (age >= VERY_STALE_DAYS)).to_numpy()
    high_vis = (flagged & (imp >= HIGH_IMPRESSIONS)).to_numpy()
    detail = np.where(very_stale & high_vis, "very_stale|high_visibility",
             np.where(very_stale, "very_stale",
             np.where(high_vis, "high_visibility", "")))

    return pd.DataFrame({
        "score": score,
        "flagged": flagged,
        "action": np.where(flagged, "review_for_refresh", "no_action"),
        "reason_code": reason,
        "reason_detail": detail,
    }, index=d.index)


res = rule(df)

view = pd.DataFrame({
    "content_id": df[COL["content_id"]],
    "client_id": df[COL["client_id"]],
    COL["impressions"]: df[COL["impressions"]],
    COL["age_days"]: df[COL["age_days"]],
    COL["position"]: df[COL["position"]],
}).join(res)

order = np.argsort(-res["score"].to_numpy(), kind="mergesort")   # stable sort: ties keep row order
ranked_df = df.iloc[order].reset_index(drop=True)                  # full original rows, same order as `out`
out = view.iloc[order].reset_index(drop=True)
out.insert(0, "rank", np.arange(1, len(out) + 1))

OUT_FILE = ROOT / "work" / "outputs" / "baseline_action_score.csv"
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUT_FILE, index=False)
print("wrote:", OUT_FILE.relative_to(ROOT), "| rows:", len(out))

print("\nflagged rows:", int(out["flagged"].sum()), "of", len(out))
display(out["reason_code"].value_counts().to_frame("rows"))


def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="mergesort")
    return np.asarray(labels)[order[:k]].mean()


# Evaluation label, same definition as the pipeline's is_declining_label: 1 when trend_direction == "down".
# It is built here, AFTER the score, and is never passed to rule().
trend = df[COL["label_src"]]
y_all = pd.Series(np.where(trend.isna(), np.nan, (trend == "down").astype(float)), index=df.index)
print("\nlabel = (trend_direction == 'down') | share 1:", round(float(y_all.mean()), 3), "| empty labels:", int(y_all.isna().sum()))
ok = y_all.notna().to_numpy()
y = y_all[ok].astype(int).to_numpy()
s = res["score"].to_numpy()[ok]
base = y.mean()
n_flag = int((s > 0).sum())

rows = []
for k in sorted({20, 50, 100, 250, 500, n_flag}):
    if 0 < k <= n_flag:                      # never read past the flagged rows - the rest is arbitrary order
        p = precision_at_k(s, y, k)
        rows.append({"K": k, "precision@K": round(p, 3), "base_rate": round(base, 3),
                     "lift_vs_base": round(p / base, 2) if base > 0 else np.nan})
if rows:
    display(pd.DataFrame(rows))
else:
    print("No flagged rows - nothing to evaluate. Re-read the rule and the input profile above.")
print(f"label rate among UNFLAGGED rows: {y[s == 0].mean():.3f}  vs  among flagged rows: {y[s > 0].mean():.3f}")
print(f"floor below the floor (always predict the majority class): accuracy = {max(base, 1 - base):.3f}")


wrote: work/outputs/baseline_action_score.csv | rows: 30000

flagged rows: 4483 of 30000


,rows
reason_code,
fresh,11518
low_visibility,10043
position_outside_range,2751
stale_visible_page1,2691
stale_visible_page2,1792
no_position_data,1205



label = (trend_direction == 'down') | share 1: 0.542 | empty labels: 0


,K,precision@K,base_rate,lift_vs_base
0,20,0.400,0.542,0.74
1,50,0.380,0.542,0.70
2,100,0.340,0.542,0.63
3,250,0.384,0.542,0.71
4,500,0.460,0.542,0.85
5,4483,0.623,0.542,1.15


label rate among UNFLAGGED rows: 0.528  vs  among flagged rows: 0.623
floor below the floor (always predict the majority class): accuracy = 0.542


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**My reading of the top 20** *(drafted from the run on the real table. Read the 20 rows yourself and edit anything you disagree with.)*

- **What they have in common (observed):** 15 of the 20 come from one client and all 20 from just 3 clients. All 20 are `keyword article`. All 20 have exactly 104 days since the last update. Positions run from 3.3 to 16.6 and impressions from 151,541 to 517,715. So the top of the queue is a small, uniform corner of the table, not a spread across it.
- **Confidence:** I trust the visibility and position numbers more (each is measured on 150k+ impressions) than the staleness signal. All 20 sit in the 104-day group, an update age shared by thousands of items, so it may come from a bulk update rather than an edit of that page. That is the main "what would make it wrong" for every row.
- **Other ways single picks could be wrong:** 8 of the 20 are at position 5 or better, so there is little room to gain. Rank 1 has 517,715 impressions, about 1.5 times rank 2, and could be driven by one big query or a spike.
- **How much I trust the list:** only as decision-support. 8 of the 20 carry the "declining" label (precision@20 = 0.400 against a base rate of 0.542), so on this proxy the top of the list is below what random picking gives (directional: 20 picks). The wider flagged set does a little better (0.623).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

TOP_N = 20
# The queue is sorted by impressions, so compare against the top-20's own median (not the whole flagged set).
med_top = float(out[COL["impressions"]].head(TOP_N).median()) if len(out) else np.nan


def confidence_note(imp, age, pos):
    vis = "higher" if imp >= HIGH_IMPRESSIONS else ("lower" if imp < 1000 else "medium")
    stale_part = ("weak - this update age is shared by 1%+ of all items, so it may be a bulk update"
                  if age in BULK_AGES else "item-specific update age")
    return (f"visibility/position confidence {vis} ({imp:,.0f} impressions, avg position {pos:.1f}, measured over 90 days); "
            f"staleness signal: {stale_part} ({age:,.0f} days); directional only")


def wrong_if(imp, age, pos, ctr_pct):
    notes, tags = [], []
    if age in BULK_AGES:
        notes.append(f"{age:,.0f} days is the update age of many items - may be a bulk update, not an edit of this page")
        tags.append("bulk_update_age")
    if pd.notna(med_top) and imp >= 3 * med_top:
        notes.append("impressions are 3x+ the typical top-20 pick - could be a one-off spike or one huge query")
        tags.append("impressions_outlier")
    if pos <= 5:
        notes.append("already close to the top 3 - little room to gain")
        tags.append("little_headroom")
    if pd.notna(ctr_pct) and ctr_pct >= 5:
        notes.append(f"ctr {ctr_pct:.2f}% (a x100 percentage) is already high - the page may be doing its job")
        tags.append("high_ctr")
    notes.append("average position blends many queries and can hide one strong and many weak ones")
    notes.append("a falling-traffic label is only a proxy for 'needs a refresh'")
    return "; ".join(notes), "|".join(tags)


if int(out["flagged"].sum()) < TOP_N:
    print(f"Warning: only {int(out['flagged'].sum())} rows are flagged, so the top {TOP_N} includes unflagged rows.")

review_rows = []
for i in range(min(TOP_N, len(out))):
    r, src = out.iloc[i], ranked_df.iloc[i]
    ctr_val = pd.to_numeric(src[COL["ctr"]], errors="coerce") if COL["ctr"] in ranked_df.columns else np.nan
    wrong_text, tags = wrong_if(r[COL["impressions"]], r[COL["age_days"]], r[COL["position"]], ctr_val)
    row = {
        "rank": int(r["rank"]),
        "content_id": r["content_id"],
        "client_id": r["client_id"],
    }
    if COL["content_type"] in ranked_df.columns:
        row["content_type"] = src[COL["content_type"]]
    row.update({
        "action": r["action"],
        "reason_code": r["reason_code"],
        "reason_detail": r["reason_detail"],
        "confidence_note": confidence_note(r[COL["impressions"]], r[COL["age_days"]], r[COL["position"]]),
        "what_would_make_it_wrong": wrong_text,
        "checks_tripped": tags,
    })
    review_rows.append(row)

top20 = pd.DataFrame(review_rows)
with pd.option_context("display.max_colwidth", None, "display.width", 250, "display.max_columns", None):
    display(top20)


,rank,content_id,client_id,content_type,action,reason_code,reason_detail,confidence_note,what_would_make_it_wrong,checks_tripped
0,1,content_5fe46e04994d,client_4e07408562,keyword article,review_for_refresh,stale_visible_page1,high_visibility,"visibility/position confidence higher (517,715 impressions, avg position 4.2, measured over 90 days); staleness signal: weak - this update age is shared by 1%+ of all items, so it may be a bulk update (104 days); directional only","104 days is the update age of many items - may be a bulk update, not an edit of this page; already close to the top 3 - little room to gain; average position blends many queries and can hide one strong and many weak ones; a falling-traffic label is only a proxy for 'needs a refresh'",bulk_update_age|little_headroom
1,2,content_2c2606c5d176,client_19581e27de,keyword article,review_for_refresh,stale_visible_page1,high_visibility,"visibility/position confidence higher (347,399 impressions, avg position 4.2, measured over 90 days); staleness signal: weak - this update age is shared by 1%+ of all items, so it may be a bulk update (104 days); directional only","104 days is the update age of many items - may be a bulk update, not an edit of this page; already close to the top 3 - little room to gain; average position blends many queries and can hide one strong and many weak ones; a falling-traffic label is only a proxy for 'needs a refresh'",bulk_update_age|little_headroom
2,3,content_cb112fce36be,client_19581e27de,keyword article,review_for_refresh,stale_visible_page1,high_visibility,"visibility/position confidence higher (309,910 impressions, avg position 5.6, measured over 90 days); staleness signal: weak - this update age is shared by 1%+ of all items, so it may be a bulk update (104 days); directional only","104 days is the update age of many items - may be a bulk update, not an edit of this page; average position blends many queries and can hide one strong and many weak ones; a falling-traffic label is only a proxy for 'needs a refresh'",bulk_update_age
3,4,content_36ff89c8214e,client_19581e27de,keyword article,review_for_refresh,stale_visible_page1,high_visibility,"visibility/position confidence higher (295,097 impressions, avg position 7.3, measured over 90 days); staleness signal: weak - this update age is shared by 1%+ of all items, so it may be a bulk update (104 days); directional only","104 days is the update age of many items - may be a bulk update, not an edit of this page; average position blends many queries and can hide one strong and many weak ones; a falling-traffic label is only a proxy for 'needs a refresh'",bulk_update_age
4,5,content_c21024970297,client_19581e27de,keyword article,review_for_refresh,stale_visible_page1,high_visibility,"visibility/position confidence higher (211,366 impressions, avg position 5.1, measured over 90 days); staleness signal: weak - this update age is shared by 1%+ of all items, so it may be a bulk update (104 days); directional only","104 days is the update age of many items - may be a bulk update, not an edit of this page; average position blends many queries and can hide one strong and many weak ones; a falling-traffic label is only a proxy for 'needs a refresh'",bulk_update_age
5,6,content_c8e9d6ab9013,client_19581e27de,keyword article,review_for_refresh,stale_visible_page1,high_visibility,"visibility/position confidence higher (208,678 impressions, avg position 9.7, measured over 90 days); staleness signal: weak - this update age is shared by 1%+ of all items, so it may be a bulk update (104 days); directional only","104 days is the update age of many items - may be a bulk update, not an edit of this page; average position blends many queries and can hide one strong and many weak ones; a falling-traffic label is only a proxy for 'needs a refresh'",bulk_update_age
6,7,content_d17681677e69,client_19581e27de,keyword article,review_for_refresh,stale_visible_page1,high_visibility,"visibility/position confidence highe

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks** (observed on the real rows. The label and `trend_pct` are used only to judge a pick after the fact.)

- **Rank 4 (`content_36ff89c8…`):** 295,097 impressions at position 7.3, but its 30-day trend is flat (+0.5%). It is a big page doing fine, flagged only because it shares the 104-day update age.
- **Rank 15 (`content_eb366e87…`):** position 16.6 and impressions up 12.4%, so there is no decline to repair. The rule cannot see trends, by design.
- **Rank 18 (`content_3430a8b9…`):** average position 3.3, barely outside the top-3 line, so little room to gain.
- **The pattern behind them:** inside the flagged set the label rate falls as impressions rise (0.720 in the lowest-impressions quartile, 0.502 in the highest). Ranking by impressions therefore puts the lowest-rate group on top (observed, directional). I did **not** change the rule because of this. It goes into the notes for the model phase.
- **Concentration:** one client holds 47% of the flagged queue and 15 of the top 20. Without that client precision@20 is 0.300 (directional: 20 picks), so one client does not look like the whole explanation.
- **For the model phase (not applied to this baseline):** use client-held-out splits (`client_id` groups). The bar to beat is 0.400 at K = 20 and 0.623 on the flagged set. Treat `days_since_last_update = 104` as a shared batch value, not an item-specific age.

**What the leakage cell below tests**

- The score reads only three columns: `impressions_90d`, `days_since_last_update`, `avg_position`. `ctr` and `content_type` appear only in the drafted review notes.
- Never read: `trend_pct`, `trend_direction` (the label source), the six 30-day comparison columns (impressions, clicks, sessions × `last_30d` / `prev_30d`, the label's inputs), `provider_used` and `model_used` (the dictionary says "not a model feature": these are the product-side fields), and the two IDs.
- The cell lists every column whose name looks like a product flag, outcome or future window, and stops with an error if the score or the notes read one. It also recomputes the score from a table that holds ONLY the three inputs and checks that the result is identical.
- **Window caveat (from the dictionary):** `impressions_90d` covers the same 90 days that contain the two 30-day windows behind the label. This table is one snapshot with no later window, so precision@K here measures how well the rule lines up with the snapshot's own "down" flag. It is **not** a forecast of what happens next.

If Run All finishes without an error, the asserts passed. Once I have read the printed lists, I confirm here in my own words: ...

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ---- weak-pick candidates (automatic checks only; a human still has to open the pages) ----
top_src = ranked_df.head(TOP_N).reset_index(drop=True)
lab20 = (top_src[COL["label_src"]] == "down").astype(float)    # evaluation label, used AFTER ranking only

client_share = top_src[COL["client_id"]].value_counts(normalize=True)
print(f"clients in the top {TOP_N}: {top_src[COL['client_id']].nunique()} | largest single-client share: {client_share.iloc[0]:.0%}")
flag_share = out.loc[out["flagged"], "client_id"].value_counts(normalize=True)
print(f"all flagged rows: {int(out['flagged'].sum())} from {flag_share.size} clients | largest client = {flag_share.iloc[0]:.0%}, "
      f"top 3 clients = {flag_share.head(3).sum():.0%} of the flagged queue")
fl_age = out.loc[out["flagged"], COL["age_days"]]
print(f"flagged rows whose update age is shared by 1%+ of all items: {fl_age.isin(BULK_AGES).mean():.0%}")
if COL["content_type"] in top_src.columns:
    print("content types in the top 20:", top_src[COL["content_type"]].value_counts().to_dict())
print(f"top {TOP_N} with label = 1: {int(lab20.sum())} of {len(lab20)}  (base rate {base:.3f}; directional only)")

why = []
for i in range(len(top20)):
    w = []
    if lab20.iloc[i] == 0:
        w.append("label says not declining (proxy only)")
    if "confidence lower" in top20.loc[i, "confidence_note"]:
        w.append("low visibility confidence")
    sharp = [t for t in top20.loc[i, "checks_tripped"].split("|") if t and t != "bulk_update_age"]  # bulk age is common; shown in the notes
    if sharp:
        w.append("checks tripped: " + "|".join(sharp))
    why.append("; ".join(w))
top20["weak_pick_candidate_because"] = why
cand = top20.loc[top20["weak_pick_candidate_because"] != "",
                 ["rank", "content_id", "reason_code", "weak_pick_candidate_because"]]
# review-only columns (trend_pct is the label's source: it is shown to JUDGE a pick afterwards, never to score one)
cand = cand.join(top_src.loc[cand.index, [COL["impressions"], COL["position"], "trend_pct"]]
                 .rename(columns={"trend_pct": "trend_pct (review only)"}))
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    display(cand)
if cand.empty:
    print("No candidates from the automatic checks - look harder: open the pages behind ranks 1-50 by hand.")

# ---- post-hoc diagnostics: these EXPLAIN the result; they are never used to change the rule ----
d = pd.DataFrame({
    "flagged": res["flagged"].to_numpy(),
    "reason": res["reason_code"].to_numpy(),
    "imp": df[COL["impressions"]].to_numpy(),
    "age": df[COL["age_days"]].to_numpy(),
    "client": df[COL["client_id"]].to_numpy(),
    "score": res["score"].to_numpy(),
    "y": y_all.to_numpy(),
})
f = d[d["flagged"]].copy()
f["impression_quartile"] = pd.qcut(f["imp"], 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
print("\nlabel rate inside the flagged set, by impressions quartile (directional):")
display(f.groupby("impression_quartile", observed=True)["y"].agg(rate="mean", rows="size").round(3))
u = d[~d["flagged"]]
print("label rate among UNFLAGGED rows, by reason code:")
display(u.groupby("reason")["y"].agg(rate="mean", rows="size").round(3))
print(f"unflagged rows that DO have position data: label rate {u.loc[u['reason'] != 'no_position_data', 'y'].mean():.3f}")
big_client = f["client"].value_counts().index[0]
rest = f[f["client"] != big_client].sort_values("score", ascending=False, kind="mergesort")
print(f"without the largest flagged client ({len(rest)} rows left): precision@20/50/100 = "
      f"{[round(float(rest['y'].head(k).mean()), 3) for k in (20, 50, 100)]}; label rate of those rows {rest['y'].mean():.3f}")
bulk = f["age"].isin(BULK_AGES)
print(f"flagged rows at a shared update age: {int(bulk.sum())} of {len(f)} (label rate {f.loc[bulk, 'y'].mean():.3f}); "
      f"other ages: {int((~bulk).sum())} rows (label rate {f.loc[~bulk, 'y'].mean():.3f}, small n)")

# ---- leakage check ----
USED_IN_SCORE = [COL["impressions"], COL["age_days"], COL["position"]]
USED_IN_NOTES = [COL[k] for k in ("ctr", "content_type") if COL[k] in df.columns]
LABEL_INPUTS = {"impressions_last_30d", "impressions_prev_30d", "clicks_last_30d",
                "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"}   # the 30-day comparison windows behind the label
NOT_FEATURES = {"provider_used", "model_used"}                                  # dictionary: "Not a model feature" (product-side fields)
NEVER_USE = ({"trend_pct", "trend_direction", "is_declining_label", COL["label_src"], COL["content_id"], COL["client_id"]}
             | LABEL_INPUTS | NOT_FEATURES)
LOOKS_RISKY = ("trend", "label", "declin", "future", "next_", "forward", "after", "post_", "last_30", "prev_30",
               "product", "provider", "model_", "flag", "outcome", "target", "refreshed")

risky_cols = [c for c in df.columns if any(p in c.lower() for p in LOOKS_RISKY)]
touched = sorted(set(USED_IN_SCORE) | set(USED_IN_NOTES))
print("\ncolumns whose NAMES look like outcome / product-flag / future fields (check each in the data dictionary):")
print(risky_cols)
print("columns the score and the notes actually read:", touched)

hit_never = sorted(set(touched) & NEVER_USE)
hit_risky = sorted(set(touched) & set(risky_cols))
assert not hit_never, f"LEAK: banned columns are read: {hit_never}"
assert not hit_risky, f"CHECK: columns that look like outcome/product/future fields are read: {hit_risky}"

only = rule(df[USED_IN_SCORE])           # the rule sees ONLY the three score columns
assert only["score"].equals(res["score"]), "score changes when other columns are removed - hidden dependency"
assert only["reason_code"].equals(res["reason_code"]), "reason codes change when other columns are removed"

date_like = [c for c in df.columns
             if re.search(r"(^|_)(date|window|period)(_|$)", c.lower()) or pd.api.types.is_datetime64_any_dtype(df[c])]
print("date / window-like columns in the table:", date_like, "| read by the score:", sorted(set(date_like) & set(touched)))
print("\nWindow note: impressions_90d is a sum over the same 90 days that contain the two 30-day windows behind the label.")
print("This table is one snapshot (no later window), so precision@K in section 2 measures agreement with the snapshot's own decline flag, not a forecast.")
print("\nLEAKAGE CHECK PASSED: no banned columns, no risky-looking columns, score unchanged when only its 3 inputs are visible.")


clients in the top 20: 3 | largest single-client share: 75%
all flagged rows: 4483 from 20 clients | largest client = 47%, top 3 clients = 87% of the flagged queue
flagged rows whose update age is shared by 1%+ of all items: 98%
content types in the top 20: {'keyword article': 20}
top 20 with label = 1: 8 of 20  (base rate 0.542; directional only)


,rank,content_id,reason_code,weak_pick_candidate_because,impressions_90d,avg_position,trend_pct (review only)
0,1,content_5fe46e04994d,stale_visible_page1,checks tripped: little_headroom,517715,4.2,-44.8
1,2,content_2c2606c5d176,stale_visible_page1,checks tripped: little_headroom,347399,4.2,-36.5
3,4,content_36ff89c8214e,stale_visible_page1,label says not declining (proxy only),295097,7.3,0.5
4,5,content_c21024970297,stale_visible_page1,label says not declining (proxy only),211366,5.1,-11.6
6,7,content_d17681677e69,stale_visible_page1,label says not declining (proxy only),201584,5.8,-15.6
7,8,content_a7427266c305,stale_visible_page1,label says not declining (proxy only),201111,5.7,-19.6
8,9,content_c5063073d048,stale_visible_page2,label says not declining (proxy only),192205,12.5,-14.6
9,10,content_3d94572c3a35,stale_visible_page1,checks tripped: little_headroom,190623,4.3,-52.4
10,11,content_01908772c6db,stale_visible_page1,checks tripped: little_headroom,187893,4.0,-20.1
11,12,content_33b4dceecad1,stale_visible_page1,label says not declining (proxy only),181574,6.2,-5.6



label rate inside the flagged set, by impressions quartile (directional):


,rate,rows
impression_quartile,,
Q1 lowest,0.720,1124
Q2,0.667,1118
Q3,0.604,1120
Q4 highest,0.502,1121


label rate among UNFLAGGED rows, by reason code:


,rate,rows
reason,,
fresh,0.581,11518
low_visibility,0.508,10043
no_position_data,0.007,1205
position_outside_range,0.608,2751


unflagged rows that DO have position data: label rate 0.554
without the largest flagged client (2367 rows left): precision@20/50/100 = [0.3, 0.36, 0.34]; label rate of those rows 0.671
flagged rows at a shared update age: 4404 of 4483 (label rate 0.619); other ages: 79 rows (label rate 0.835, small n)

columns whose NAMES look like outcome / product-flag / future fields (check each in the data dictionary):
['provider_used', 'model_used', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'trend_direction', 'trend_pct']
columns the score and the notes actually read: ['avg_position', 'content_type', 'ctr', 'days_since_last_update', 'impressions_90d']
date / window-like columns in the table: [] | read by the score: []

Window note: impressions_90d is a sum over the same 90 days that contain the two 30-day windows behind the label.
This table is one snapshot (no later window), so precision@K in section 2 measures 

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.